# Exercise 7 — Hardware validation of two 4-bit registers (PYNQ-Z2)

**Target:** PYNQ-Z2 (Zynq-7020) · Vivado PS/PL design · AXI GPIO controlled from Python.

This notebook tests the **parallel 4-bit register**, **4-bit serial-in/parallel-out shift register (SIPO)** and **output multiplexer** implemented in `register_system.vhd`. It replaces the previous experimental `register_test.ipynb` with a reproducible procedure.

### Hardware path

```text
PYNQ Python / ARM PS (Linux)
        | MMIO (M_AXI_GP0)
        v
AXI interconnect --> axi_gpio_0, base 0x41200000
                        | CH1: 9 output command bits
                        v
                register_command_adapter
                        | load, shift, mode, data, serial
               +--------+--------+
               v                 v
        parallel_register   shift_register
               +--------+--------+
                        | mode selects q[3:0]
                        v
                 AXI GPIO CH2 (4 inputs)
                        |
                        v
                 Python: q[3:0]
```

### Confirmed design parameters

| Item | Value |
| --- | --- |
| AXI GPIO base / MMIO range | `0x41200000` / `0x1000` |
| CH1 `DATA`, `TRI` | `0x00`, `0x04` |
| CH2 `DATA`, `TRI` | `0x08`, `0x0C` |
| Channel widths | 9 command outputs / 4 result inputs |
| Vivado `FCLK_CLK0` / timing target | **50 MHz** |
| Parallel output selection | `mode=0` |
| SIPO output selection | `mode=1` |
| Software reset command bit 8 | **Not wired** to `register_system.reset` |

> **Read carefully.** An access to an unresponsive PL AXI address previously made Linux, SSH and Jupyter unresponsive. Python `timeout`, a notebook interrupt or a `try/finally` block **cannot reliably recover** a stalled MMIO transaction. A power cycle may be required. Program the correct `.bit` and matching `.ltx` from Vivado first. The notebook **does not program the FPGA**. Run each cell in order and stop at the first unexpected result. **Do not** run all cells automatically on an unverified board.

Successful exercise-7 hardware results were previously observed on **2026-11-08**: parallel `0xA`, SIPO `0xB`. Those past results are *not* preloaded as notebook outputs; the tests below must run on the board to reproduce them.

## Step 1 — Software constants (no PL access)

Hardware changes required before running any MMIO cell:

1. Boot the PYNQ-Z2 and confirm Jupyter/SSH are responsive.
2. In Vivado Hardware Manager, **Program Device** with the exercise-7 `design_1_wrapper.bit` and matching `design_1_wrapper.ltx` (or otherwise establish that the exact same design is loaded).
3. Check that the Block Design provides `M_AXI_GP0` to AXI GPIO at `0x41200000` with the intended resets and clocks.
4. Confirm no conflicting notebook or program is driving this GPIO.

The explicit `HARDWARE_READY` flag is a deliberate guard, not automatic hardware detection. Set it to `True` **only after** the checks above.

In [ ]:
import time
from datetime import datetime, timezone
from pynq import Clocks, MMIO

BASE_ADDR = 0x41200000
ADDR_RANGE = 0x1000
FCLK0_MHZ = 50.0

# AXI GPIO register offsets
GPIO_DATA  = 0x00
GPIO_TRI   = 0x04
GPIO2_DATA = 0x08
GPIO2_TRI  = 0x0C

# Command layout in register_command_adapter.vhd
DATA_MASK    = 0x00F
SERIAL_IN    = 0x010  # bit 4
MODE         = 0x020  # bit 5: 0=parallel, 1=shift
LOAD         = 0x040  # bit 6: rising-edge request
SHIFT        = 0x080  # bit 7: rising-edge request
RESET        = 0x100  # bit 8: NOT CONNECTED to register_system/reset
COMMAND_MASK = 0x1FF

# GPIO command pacing; not the FPGA clock period.
COMMAND_DELAY_SECONDS = 0.010

HARDWARE_READY = True  # MANUALLY change to True after verifying the loaded design

results = {}  # Populated by tests actually run in this kernel.
print(f"AXI GPIO: 0x{BASE_ADDR:08X}, range: 0x{ADDR_RANGE:X}")
print("Hardware confirmed by operator:", HARDWARE_READY)

## Step 2 — Confirm hardware readiness and configure FCLK0

After a board restart, PYNQ previously reported **100 MHz** while this Vivado implementation was constrained for **50 MHz**. In the observed setup, `Clocks.fclk0_mhz = 50.0` changed the PS clock register, even though Linux debugfs continued to report a stale `100 MHz` value. Use the PYNQ clock reading as the immediate runtime check; the SLCR register can be examined separately if needed.

**Changing FCLK0 affects running PL logic and the ILA/debug hub.** Ensure that the programmed bitstream is the intended one before doing so. This step only touches clock configuration, not AXI GPIO.

In [8]:
if not HARDWARE_READY:
    raise RuntimeError(
        "Stop: verify the exercise-7 bitstream, address map and hardware state, "
        "then set HARDWARE_READY = True in Step 1 and rerun Step 1."
    )

before = Clocks.fclk0_mhz
print(f"FCLK0 before: {before:.2f} MHz")
if abs(before - FCLK0_MHZ) > 0.01:
    Clocks.fclk0_mhz = FCLK0_MHZ

after = Clocks.fclk0_mhz
print(f"FCLK0 after:  {after:.2f} MHz")
if abs(after - FCLK0_MHZ) > 0.01:
    raise RuntimeError("FCLK0 is not at the expected 50 MHz; do not access the AXI GPIO.")

clock_ok = True
print("Clock preparation: PASS")

FCLK0 before: 50.00 MHz
FCLK0 after:  50.00 MHz
Clock preparation: PASS


## Step 3 — Initialize MMIO (no read/write yet)

Creating `MMIO(...)` maps the address range. This **does not prove** that the slave responds. The first actual access happens in Step 4.

In [9]:
if not HARDWARE_READY or not globals().get("clock_ok", False):
    raise RuntimeError("Complete hardware and clock checks before creating MMIO.")

mmio = MMIO(BASE_ADDR, ADDR_RANGE)
print(f"MMIO mapped at 0x{BASE_ADDR:08X}")

MMIO mapped at 0x41200000


## Step 4 — First live AXI reads: GPIO direction diagnostics

**These are the first AXI GPIO transactions.** If the cell stops responding, do not re-execute it. Do not interpret the lack of a Python exception as proof that MMIO access cannot hang.

On this board, both `GPIO_TRI` reads returned `0xFFFFFFFF` even though the IP was configured with **`C_ALL_OUTPUTS=1` for CH1** and **`C_ALL_INPUTS_2=1` for CH2**, and both register tests later passed. Therefore **do not assert** that the returned TRI values must match the software-configurable direction bits. Record these readings as diagnostics; the Vivado IP configuration is authoritative for the fixed-direction ports.

Do **not** write the TRI registers as part of the functional test.

In [10]:
direction1 = mmio.read(GPIO_TRI)
direction2 = mmio.read(GPIO2_TRI)

print(f"CH1 TRI: 0x{direction1:08X}")
print(f"CH2 TRI: 0x{direction2:08X}")
print("Direction register values recorded (not a fixed-direction PASS/FAIL check).")
results["gpio_tri_read"] = "PASS"

CH1 TRI: 0xFFFFFFFF
CH2 TRI: 0xFFFFFFFF
Direction register values recorded (not a fixed-direction PASS/FAIL check).


## Step 5 — Command encoding and helper functions

```text
commands[8:0] = [RESET][SHIFT][LOAD][MODE][SERIAL_IN][D3][D2][D1][D0]
                    8      7     6    5      4         3..0

RESET=0x100  (NOT CONNECTED to register_system reset)
SHIFT=0x080  LOAD=0x040  MODE=0x020  SERIAL_IN=0x010
```

The VHDL command adapter synchronizes `LOAD` / `SHIFT` and converts a **0→1 transition** into a one-clock pulse. Each operation therefore writes: *prepare with request low → request high → request low*. The 10 ms pauses are for stable separate CPU writes, not a replacement for the HDL synchronizer.

A `finally` block tries to leave requests inactive if ordinary Python exceptions occur; it **cannot** run while a kernel-level MMIO access is stuck. Neither the `RESET` bit nor writing zero to CH1 is a general hardware reset. The SIPO test uses four shifts, which completely replace its previous four stored bits.

In [11]:
def write_commands(value: int) -> None:
    if not isinstance(value, int) or not 0 <= value <= COMMAND_MASK:
        raise ValueError(f"Command must be an integer in [0, 0x{COMMAND_MASK:X}]")
    mmio.write(GPIO_DATA, value)
    time.sleep(COMMAND_DELAY_SECONDS)


def read_register() -> int:
    return mmio.read(GPIO2_DATA) & DATA_MASK


def load_parallel(value: int) -> int:
    if not isinstance(value, int) or not 0 <= value <= DATA_MASK:
        raise ValueError("Parallel value must be a 4-bit integer")
    base = value  # mode=0, no LOAD/SHIFT assertion
    try:
        write_commands(base)
        write_commands(base | LOAD)
    finally:
        write_commands(base)  # release LOAD on ordinary Python errors
    return read_register()


def shift_bit(bit: int) -> int:
    if bit not in (0, 1):
        raise ValueError("Serial bit must be 0 or 1")
    base = MODE | (SERIAL_IN if bit else 0)
    try:
        write_commands(base)
        write_commands(base | SHIFT)
    finally:
        write_commands(base)  # release SHIFT on ordinary Python errors
    return read_register()


def select_mode(mode: int) -> int:
    if mode not in (0, 1):
        raise ValueError("mode must be 0 (parallel) or 1 (SIPO)")
    write_commands(MODE if mode else 0)
    return read_register()

print("Command helpers defined; no GPIO transactions performed in this cell.")

Command helpers defined; no GPIO transactions performed in this cell.


## Step 6 — Test parallel register (expected `0xA`)

This is the first **functional write** to the PL. It sends `0x00A → 0x04A → 0x00A` on CH1 and reads CH2. If a write stalls, do not restart the cell or issue more MMIO commands.

In [12]:
expected_parallel = 0xA
actual_parallel = load_parallel(expected_parallel)
results["parallel"] = "PASS" if actual_parallel == expected_parallel else "FAIL"

print("PARALLEL REGISTER")
print(f"Expected: 0x{expected_parallel:X} ({expected_parallel:04b})")
print(f"Actual:   0x{actual_parallel:X} ({actual_parallel:04b})")
print(f"Result:   {results['parallel']}")
assert actual_parallel == expected_parallel, "Parallel register mismatch"

PARALLEL REGISTER
Expected: 0xA (1010)
Actual:   0xA (1010)
Result:   PASS


## Step 7 — Test SIPO shift register (expected `0xB`)

Inject the bits `1, 0, 1, 1`. Intermediate states **may depend on the initial register content**, because software reset is not connected. After four shifts, however, all four old bits are replaced, so the **final result must be `1011`** regardless of initial state.

No software reset is attempted here.

In [13]:
sequence = [1, 0, 1, 1]
expected_shift = 0xB
shift_states = []

print("SIPO SHIFT REGISTER")
for bit in sequence:
    value = shift_bit(bit)
    shift_states.append(value)
    print(f"Bit injected: {bit} -> Register: {value:04b}")

actual_shift = shift_states[-1]
results["sipo"] = "PASS" if actual_shift == expected_shift else "FAIL"
print(f"Expected: 0x{expected_shift:X}")
print(f"Actual:   0x{actual_shift:X}")
print(f"Result:   {results['sipo']}")
assert actual_shift == expected_shift, "SIPO register mismatch"

SIPO SHIFT REGISTER
Bit injected: 1 -> Register: 0111
Bit injected: 0 -> Register: 1110
Bit injected: 1 -> Register: 1101
Bit injected: 1 -> Register: 1011
Expected: 0xB
Actual:   0xB
Result:   PASS


## Step 8 — Test output selection / independent storage

This step expects the **parallel register to contain `0xA`** and the **SIPO register to contain `0xB`**, as prepared by Steps 6 and 7. It checks that changing `mode` selects the output without issuing `LOAD` or `SHIFT` again.

To run this step independently, first rerun Steps 6 and 7.

In [14]:
if results.get("parallel") != "PASS" or results.get("sipo") != "PASS":
    raise RuntimeError("Run and pass parallel and SIPO tests before output-selection test")

parallel_selected = select_mode(0)
shift_selected = select_mode(1)
results["output_selection"] = (
    "PASS" if (parallel_selected, shift_selected) == (0xA, 0xB) else "FAIL"
)

print(f"Parallel selected: 0x{parallel_selected:X} ({parallel_selected:04b})")
print(f"SIPO selected:     0x{shift_selected:X} ({shift_selected:04b})")
print(f"Selection result:  {results['output_selection']}")
assert (parallel_selected, shift_selected) == (0xA, 0xB), "Output selection mismatch"

# Leave LOAD/SHIFT low and select parallel output.
write_commands(0)

Parallel selected: 0xA (1010)
SIPO selected:     0xB (1011)
Selection result:  PASS


## Step 9 — Summary and reproducibility record

This cell does not perform MMIO. It reports **only tests actually completed in the current notebook kernel**, not the successful historical results from 2026-11-08.

In [15]:
print("EXERCISE 7 — HARDWARE TEST SUMMARY")
print("UTC time:", datetime.now(timezone.utc).isoformat(timespec="seconds"))
print("Board: PYNQ-Z2 / Zynq-7020")
print(f"AXI GPIO base: 0x{BASE_ADDR:08X}")
print(f"Target FCLK0: {FCLK0_MHZ:.1f} MHz")
for key, label in [
    ("gpio_tri_read", "CH1/CH2 TRI diagnostics"),
    ("parallel", "4-bit parallel register"),
    ("sipo", "4-bit SIPO shift register"),
    ("output_selection", "Output selection / storage"),
]:
    print(f"{label:32} {results.get(key, 'NOT RUN')}")

required = ("parallel", "sipo", "output_selection")
overall = "PASS" if all(results.get(x) == "PASS" for x in required) else "INCOMPLETE / FAIL"
print("Overall functional result:", overall)
print("Software reset: NOT TESTED (not connected in this design)")

EXERCISE 7 — HARDWARE TEST SUMMARY
UTC time: 2026-11-08T21:19:54+00:00
Board: PYNQ-Z2 / Zynq-7020
AXI GPIO base: 0x41200000
Target FCLK0: 50.0 MHz
CH1/CH2 TRI diagnostics          PASS
4-bit parallel register          PASS
4-bit SIPO shift register        PASS
Output selection / storage       PASS
Overall functional result: PASS
Software reset: NOT TESTED (not connected in this design)


## Troubleshooting notes and known limitations

### If MMIO hangs

- **Do not retry** the same cell, issue more MMIO accesses, or assume that notebook interruption can cancel a stalled hardware access.
- The problem may require physically rebooting the PYNQ-Z2. If Vivado Hardware Manager/JTAG interferes with reboot, disconnect the target before restarting.
- After reboot, confirm **the correct bitstream is programmed again**, then recheck FCLK0 (previously it returned to 100 MHz after boot).
- To investigate recurrent failures, arm a **System ILA** on the AXI interface *before* a controlled test: `ARVALID=1` for read requests, `AWVALID=1` for writes. Inspect `AR/ R` and `AW / W / B` handshake signals respectively. Capturing a later successful transaction does not prove why an earlier one hung.

### Confirmed vs unresolved (2026-11-08 investigation)

- **Confirmed:** `GPIO_TRI` reads completed, a standalone `GPIO_DATA` write returned and was observed with an AXI `BRESP=OKAY`, parallel loading `0xA` passed, and four SIPO shifts produced `0xB`.
- **Confirmed:** runtime FCLK0 was initially 100 MHz on boot, then set to 50 MHz in PYNQ. The SLCR `FPGA0_CLK_CTRL` register was observed as `0x00101400` while PYNQ reported 50 MHz; Linux debugfs still displayed 100 MHz.
- **Confirmed:** `proc_sys_reset.dcm_locked` was connected high, and the AXI active-low resets were captured deasserted.
- **Not established:** the **unique root cause** of prior freezes. Clock/reset changes and the final passing result were not isolated in controlled A/B experiments. Some earlier cells combined multiple MMIO operations, so the exact stalled access was not always known.
- **Not available:** command bit 8 (`RESET`) does not reach `register_system/reset`. The registers use an active-high asynchronous reset connected to `rst_ps7_0_50M/peripheral_reset`; adding software reset requires a deliberate hardware-design change.

### Version-controlled evidence

Keep the matching Vivado Block Design, RTL, bitstream generation instructions, timing constraints and screenshots/ILA captures together with the issue follow-up. **Do not commit a claim of a permanent MMIO hang fix without reproducible evidence.**

Reference: [fpga_lab issue #1](https://github.com/dexterBEN/fpga_lab/issues/1).